# Lab 06-03: Build a Custom MCP Server for a SaaS Tool

**Book chapter:** Chapter 06: Enterprise Knowledge: RAG, MCP, and Integrations  
**Estimated time:** 25 minutes

## Overview

Cymbal Retail's support tickets live in a SaaS ticketing system that has no connector.
Several agents need to read tickets, but only the support agent may create them. In
this lab, you build an MCP server that wraps the ticketing API with three task-level
tools and input validation, and connect two agents to it with different tools.

## Learning Objectives

By the end of this lab, you will be able to:

- Build an MCP server with task-level tools
- Validate tool inputs and return actionable errors
- Connect ADK agents to an MCP server over stdio
- Limit an agent's tools with a tool filter

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents for Gemini requests. The MCP server runs locally.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Practice | How the lab applies it |
|---|---|
| Task-level tools | Search, get, and create tickets, instead of raw API endpoints |
| Input validation | Emails, priorities, and summary lengths are checked |
| Actionable errors | `ToolError` messages tell the agent what to fix |
| Least privilege | A tool filter gives the status agent read tools only |

## Setup

In [ ]:
%pip install --quiet --upgrade google-adk mcp

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The lab code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Create the MCP Server

The next cell writes `ticket_server.py`. A dictionary stands in for the SaaS API so that
the lab needs no account. `MCPServer` turns each decorated function into an MCP tool, and
its docstring becomes the tool's description. Raising `ToolError` sends the message to the
agent so that it can correct itself.

> **Note:** `MCPServer` was named `FastMCP` in versions of the MCP Python SDK before 2.0.

In [ ]:
%%writefile ticket_server.py
"""A custom MCP server that wraps a ticketing SaaS API."""
import re

from mcp.server.mcpserver import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

mcp = MCPServer("cymbal-tickets")

# This dictionary stands in for the SaaS ticketing API.
TICKETS = {
    "T-1001": {"email": "maria@example.com", "status": "open",
               "priority": "normal", "summary": "Boot sole is peeling"},
    "T-1002": {"email": "maria@example.com", "status": "closed",
               "priority": "low", "summary": "Change delivery address"},
}
EMAIL = re.compile(r"^[^@\s]+@[^@\s]+\.[a-z]{2,}$")


def check_email(email):
    """Rejects anything that is not a plain email address.

    ToolError sends its message to the agent so that it can correct itself.
    """
    if not EMAIL.match(email):
        raise ToolError("customer_email must be a valid email address")


@mcp.tool()
def search_tickets(customer_email: str) -> list[dict]:
    """Lists the support tickets opened by one customer."""
    check_email(customer_email)
    return [{"ticket_id": ticket_id, "status": t["status"],
             "summary": t["summary"]}
            for ticket_id, t in TICKETS.items()
            if t["email"] == customer_email]


@mcp.tool()
def get_ticket(ticket_id: str) -> dict:
    """Returns the details of one support ticket, such as T-1001."""
    if ticket_id not in TICKETS:
        raise ToolError(f"Ticket {ticket_id} does not exist")
    return {"ticket_id": ticket_id, **TICKETS[ticket_id]}


@mcp.tool()
def create_ticket(customer_email: str, summary: str,
                  priority: str) -> dict:
    """Opens a support ticket. Priority must be low, normal, or high."""
    check_email(customer_email)
    if priority not in ("low", "normal", "high"):
        raise ToolError("priority must be low, normal, or high")
    if not 5 <= len(summary) <= 200:
        raise ToolError("summary must be 5 to 200 characters long")
    ticket_id = f"T-{1001 + len(TICKETS)}"
    TICKETS[ticket_id] = {"email": customer_email, "status": "open",
                          "priority": priority, "summary": summary}
    return {"ticket_id": ticket_id, "status": "open"}


if __name__ == "__main__":
    mcp.run()  # Communicate over standard input and output (stdio).

## Step 1: Define a helper that connects an agent to the server

`McpToolset` starts `ticket_server.py` as a separate process and talks to it over
standard input and output (stdio). `tool_filter` lists the tools an agent may use; with
no filter, the agent receives every tool. The `chat` helper prints each tool call
(`->`), each result (`<-`), and the agent's answer.

In [ ]:
import os
import sys

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from google.genai import types
from mcp import StdioServerParameters

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")


def ticket_tools(allowed=None):
    """Starts ticket_server.py over stdio and exposes the allowed tools."""
    return McpToolset(
        connection_params=StdioConnectionParams(
            server_params=StdioServerParameters(
                command=sys.executable, args=["ticket_server.py"]),
            timeout=30),
        tool_filter=allowed)


def chat(agent, text):
    """Sends one message and prints the tool calls and the answer."""
    runner = Runner(app_name="cymbal", agent=agent,
                    auto_create_session=True,
                    session_service=InMemorySessionService())
    print(f"\n[{agent.name}] User: {text}")
    message = types.Content(role="user", parts=[types.Part(text=text)])
    for event in runner.run(user_id="agent", session_id="s1",
                            new_message=message):
        for call in event.get_function_calls():
            print(f"  -> {call.name}({dict(call.args)})")
        for reply in event.get_function_responses():
            print(f"  <- {str(reply.response)[:90]}")
        if event.is_final_response() and event.content and \
                event.content.parts:
            print("Agent:", event.content.parts[0].text)

## Step 2: Give a status agent read-only tools with a tool filter

The status agent receives only the two read tools.

In [ ]:
status_agent = Agent(
    name="ticket_status",
    model=MODEL,
    instruction="You report on existing support tickets. Use the tools, "
                "and never invent ticket details.",
    tools=[ticket_tools(allowed=["search_tickets", "get_ticket"])],
)
chat(status_agent, "What tickets does maria@example.com have open?")
chat(status_agent, "Open a new ticket for maria@example.com about a late "
                   "refund.")

**Expected result:** the agent lists Maria's tickets, but explains that it cannot open a ticket, because `create_ticket` is not among its tools.

## Step 3: Give a support agent every tool, including create_ticket

The support agent receives every tool, including `create_ticket`.

In [ ]:
support_agent = Agent(
    name="ticket_support",
    model=MODEL,
    instruction="You manage support tickets. Use the tools, and explain "
                "any error that a tool returns.",
    tools=[ticket_tools()],
)
chat(support_agent, "Open a high priority ticket for maria@example.com: "
                    "refund for order 10422 has not arrived.")
chat(support_agent, "Open an urgent ticket for maria@example.com: the "
                    "jacket zipper broke.")

**Expected results:** a new high priority ticket with an ID such as `T-1003`, and an
error for the urgent ticket that says priority must be low, normal, or high. The
agent explains the error instead of guessing.

## Check Your Understanding

**1. Why does the status agent fail to open a ticket?**

<details><summary>Show answer</summary>

Its tool filter includes only search_tickets and get_ticket, so create_ticket is not available to it.

</details>

**2. Why does the server raise ToolError instead of ValueError?**

<details><summary>Show answer</summary>

ToolError messages are sent to the agent. Other exceptions are reported only as a generic error, which the agent cannot act on.

</details>

**3. When is a custom MCP server better than a function tool?**

<details><summary>Show answer</summary>

When several agents or clients need the same integration, or when the team that owns the system should own and deploy the integration separately.

</details>

## Summary

You built an MCP server with validated, task-level tools and connected two agents with
different tools. In production, the server would run on Cloud Run with its API credentials
in Secret Manager, and it would be registered in Agent Registry.